# Qwen3-4B Accounting & TDS Assessment — Standalone

Consumes the complete Stage-1 invoice JSON from Qwen3-VL plus a supplied Chart of Accounts.

Separates **Task A (COA / Accounting Classification)** and **Task B (TDS Assessment)** into two focused reasoning calls using the quantized Qwen3-4B model.

Removes all deterministic financial math, GST calculations, and reconciliation from the LLM.

## 1. Install dependencies

In [ ]:
!pip install -q -U "transformers>=4.51" accelerate bitsandbytes pydantic fastapi uvicorn nest_asyncio pyngrok

## 2. Imports & Setup

In [ ]:
import copy
import json
import re
import traceback
from typing import Optional, List, Dict, Any

import torch
from pydantic import BaseModel, Field
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

torch.manual_seed(0)

## 3. Load Quantized Qwen3-4B Model (4-bit)

In [ ]:
MODEL_NAME_TEXT = "Qwen/Qwen3-4B-Instruct-2507"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

text_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME_TEXT)
text_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME_TEXT,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
text_model.eval()
print("Loaded:", MODEL_NAME_TEXT)
print("4-bit quantization: enabled")

## 4. Helper Utilities

In [ ]:
def _as_float(value):
    if value is None or value == "":
        return None
    try:
        return float(value)
    except (TypeError, ValueError):
        return None

def _clean_numeric(value):
    if value is None or value == "":
        return None
    if isinstance(value, (int, float)):
        return float(value)
    text = str(value).strip().replace(",", "")
    text = re.sub(r"^(?:Rs\.?|INR|₹)\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s*/-\s*$", "", text)
    negative = text.startswith("(") and text.endswith(")")
    text = text.strip("()")
    try:
        number = float(text)
        return -number if negative else number
    except ValueError:
        return None

def safe_json_parse(text):
    if not isinstance(text, str):
        return None
    cleaned = re.sub(r"^```(?:json)?\s*|\s*```$", "", text.strip(), flags=re.IGNORECASE | re.MULTILINE)
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        start, end = cleaned.find("{"), cleaned.rfind("}")
        if start >= 0 and end > start:
            try:
                return json.loads(cleaned[start:end + 1])
            except json.JSONDecodeError:
                return None
        return None

## 5. Task A — COA / Accounting Classification

Classifies every invoice line item against the supplied Chart of Accounts using the complete invoice context.

In [ ]:
def classify_coa_accounting(
    complete_invoice_json: dict,
    chart_of_accounts: list,
) -> list:
    """
    Task A: Classifies each line item against the supplied Chart of Accounts.
    
    The model receives:
      - COMPLETE invoice JSON (vendor, customer, totals, tax, bank, all line items)
      - Supplied Chart of Accounts list
      
    Responsibilities:
      - Select exactly one COA account for every non-empty line item
      - Provide confidence score and concise accounting reason
      - Flag uncertain items for review
      
    Strictly NOT responsible for:
      - GST calculations, line math, subtotal/total reconciliation, or journal arithmetic.
    """
    if not isinstance(complete_invoice_json, dict):
        raise TypeError("complete_invoice_json must be a dictionary")

    line_items = complete_invoice_json.get("line_items") or []
    chart_of_accounts = chart_of_accounts or []

    # Map supplied Chart of Accounts
    accounts_map = {}
    coa_lines = []
    for idx, acc in enumerate(chart_of_accounts, 1):
        acc_id = str(acc.get("account_id") or acc.get("id") or f"ACC_{idx}").strip()
        acc_name = str(acc.get("account_name") or acc.get("name") or "Uncategorized").strip()
        acc_type = str(acc.get("account_type") or acc.get("type") or "expense").strip()
        accounts_map[acc_id] = (acc_id, acc_name)
        accounts_map[acc_name.lower()] = (acc_id, acc_name)
        coa_lines.append({
            "account_id": acc_id,
            "account_name": acc_name,
            "account_type": acc_type,
        })

    system_prompt = """You are an expert Indian accounting classification engine.

TASK:
Classify every line item from the supplied invoice against the supplied Chart of Accounts (COA).

INPUT:
You receive the COMPLETE invoice JSON and the live Chart of Accounts.
Read the entire invoice context (vendor, customer, nature of business, amounts, taxes, line descriptions, HSN/SAC).

ACCOUNTING RULES:
1. Select exactly one account for each line item strictly from the supplied Chart of Accounts.
2. The account_id and account_name MUST come from the supplied COA list.
3. Never invent, rename, or create a new account.
4. GST, TDS, and other tax amounts are NOT expense account categories.
5. Use line description, HSN/SAC code, quantities, unit prices, vendor name, and invoice context as evidence.
6. Provide a confidence_score between 0.0 and 1.0.
7. Provide a clear, concise accounting_reason explaining why the COA category was selected.
8. If the classification is genuinely uncertain, flag ai_needs_review: true.

DO NOT PERFORM:
- GST arithmetic
- Line-item math validation
- Subtotal/tax/total reconciliation
- Financial reconciliation or balancing

OUTPUT FORMAT:
Return ONLY valid JSON with no markdown formatting or extra text:
{
  "accounting": [
    {
      "line_index": 1,
      "source_description": "...",
      "account_id": "...",
      "account_name": "...",
      "confidence_score": 0.97,
      "ai_needs_review": false,
      "accounting_reason": "..."
    }
  ]
}
Return one entry for EVERY input line item in the SAME order.
"""

    user_payload = {
        "invoice_json": complete_invoice_json,
        "chart_of_accounts": coa_lines,
    }

    messages = [
        {"role": "system", "content": system_prompt},
        {
            "role": "user",
            "content": (
                "Classify the line items from this COMPLETE invoice against the supplied Chart of Accounts:\n\n"
                + json.dumps(user_payload, ensure_ascii=False, indent=2)
            ),
        },
    ]

    try:
        text_prompt = text_tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
        inputs = text_tokenizer([text_prompt], return_tensors="pt").to(text_model.device)
        pad_id = text_tokenizer.pad_token_id or text_tokenizer.eos_token_id

        with torch.inference_mode():
            outputs = text_model.generate(
                **inputs,
                max_new_tokens=max(1024, 250 * max(1, len(line_items))),
                do_sample=False,
                pad_token_id=pad_id,
            )

        generated = outputs[0][inputs.input_ids.shape[1]:]
        raw_output = text_tokenizer.decode(generated, skip_special_tokens=True).strip()
        parsed = safe_json_parse(raw_output)

        if not isinstance(parsed, dict) or not isinstance(parsed.get("accounting"), list):
            raise ValueError("Invalid accounting classification JSON from model")

        raw_accounting = parsed.get("accounting", [])
        normalized_results = []

        for index, item in enumerate(line_items, 1):
            model_item = raw_accounting[index - 1] if index - 1 < len(raw_accounting) else {}
            if not isinstance(model_item, dict):
                model_item = {}

            description = str(
                item.get("description")
                or item.get("product_name")
                or item.get("name")
                or model_item.get("source_description")
                or ""
            ).strip()

            matched_id = str(model_item.get("account_id") or "").strip()
            matched_name = str(model_item.get("account_name") or "").strip()

            if matched_id in accounts_map:
                valid_id, valid_name = accounts_map[matched_id]
            elif matched_name.lower() in accounts_map:
                valid_id, valid_name = accounts_map[matched_name.lower()]
            else:
                valid_id, valid_name = None, None

            try:
                confidence = float(model_item.get("confidence_score") or 0.0)
            except (TypeError, ValueError):
                confidence = 0.0
            confidence = max(0.0, min(1.0, confidence))

            needs_review = bool(
                valid_id is None
                or confidence < 0.70
                or model_item.get("ai_needs_review") is True
            )

            normalized_results.append({
                "line_index": index,
                "source_description": description,
                "account_id": valid_id,
                "account_name": valid_name,
                "confidence_score": round(confidence, 2),
                "ai_needs_review": needs_review,
                "accounting_reason": model_item.get("accounting_reason") or "Classified based on item description and invoice context.",
            })

        return normalized_results

    except Exception as exc:
        print(f"[classify_coa_accounting] Error: {exc}")
        traceback.print_exc()
        fallback = []
        for index, item in enumerate(line_items, 1):
            fallback.append({
                "line_index": index,
                "source_description": (item or {}).get("description", "") if isinstance(item, dict) else "",
                "account_id": None,
                "account_name": None,
                "confidence_score": 0.0,
                "ai_needs_review": True,
                "accounting_reason": f"Classification fallback due to error: {str(exc)}",
            })
        return fallback

## 6. Task B — TDS Assessment

Determines whether TDS appears applicable, the likely section and rate, and provides reasoning without performing final arithmetic.

In [ ]:
def assess_tds_reasoning(
    complete_invoice_json: dict,
) -> dict:
    """
    Task B: Performs focused TDS (Tax Deducted at Source) assessment.
    
    The model receives:
      - COMPLETE invoice JSON (vendor, customer, PAN, line items, amounts, taxes, terms)
      
    Responsibilities:
      - Assess whether TDS appears applicable
      - Identify likely TDS section (e.g. 194C, 194J, 194H, 194I, 194Q)
      - Propose likely TDS rate (%) and base amount
      - Provide proposed TDS amount as an AI assessment
      - Flag uncertain situations for review with clear reasoning
      
    IMPORTANT:
      - The proposed TDS amount is an assessment only; deterministic calculations
        (thresholds, cumulative YTD, rules) will be handled by the backend Finance engine.
    """
    if not isinstance(complete_invoice_json, dict):
        raise TypeError("complete_invoice_json must be a dictionary")

    system_prompt = """You are an expert Indian TDS (Tax Deducted at Source) assessment engine.

TASK:
Analyze the supplied COMPLETE invoice JSON and determine TDS applicability under Indian Income Tax regulations.

INPUT:
You receive the complete invoice JSON, including vendor details, customer/payer details, PANs, HSN/SAC codes, line-item descriptions, payment terms, and amounts.

TDS ASSESSMENT RULES:
1. Assess whether TDS is potentially applicable based on the nature of payment (e.g., professional/technical fees under 194J, contractor/works under 194C, rent under 194I, commission/brokerage under 194H, purchase of goods under 194Q).
2. If TDS is explicitly stated on the invoice, preserve the extracted section, rate, and base amount.
3. If TDS is inferred from context, determine the most likely TDS provision/section and standard rate.
4. Determine the proposed TDS base amount (typically subtotal / taxable amount before GST).
5. If both rate and base amount are identified, propose the estimated TDS amount (base * rate / 100).
6. Do NOT fabricate or invent a section/rate if the invoice does not provide sufficient context; return null for uncertain fields and set tds_needs_review: true.
7. Explain your assessment concisely in tds_reasoning.

DO NOT PERFORM:
- Complex threshold arithmetic (e.g. tracking annual PAN thresholds)
- Catch-up or year-to-date arithmetic
- Statutory final reconciliation

OUTPUT FORMAT:
Return ONLY valid JSON with no markdown formatting or extra text:
{
  "tds_assessment": {
    "tds_applicable": true,
    "tds_provision": "194J",
    "tds_rate": 10.0,
    "tds_base_amount": 50000.0,
    "proposed_tds_amount": 5000.0,
    "tds_needs_review": false,
    "tds_reasoning": "Professional/technical consulting services are subject to TDS under Section 194J at 10%."
  }
}
"""

    messages = [
        {"role": "system", "content": system_prompt},
        {
            "role": "user",
            "content": (
                "Perform TDS assessment on this COMPLETE invoice JSON:\n\n"
                + json.dumps(complete_invoice_json, ensure_ascii=False, indent=2)
            ),
        },
    ]

    try:
        text_prompt = text_tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
        inputs = text_tokenizer([text_prompt], return_tensors="pt").to(text_model.device)
        pad_id = text_tokenizer.pad_token_id or text_tokenizer.eos_token_id

        with torch.inference_mode():
            outputs = text_model.generate(
                **inputs,
                max_new_tokens=512,
                do_sample=False,
                pad_token_id=pad_id,
            )

        generated = outputs[0][inputs.input_ids.shape[1]:]
        raw_output = text_tokenizer.decode(generated, skip_special_tokens=True).strip()
        parsed = safe_json_parse(raw_output)

        if not isinstance(parsed, dict) or "tds_assessment" not in parsed:
            if isinstance(parsed, dict) and ("tds_applicable" in parsed or "applicable" in parsed):
                assessment = parsed
            else:
                raise ValueError("Stage 3 TDS output is not a valid JSON object")
        else:
            assessment = parsed["tds_assessment"]

        # Clean numeric fields
        rate = _clean_numeric(assessment.get("tds_rate"))
        base = _clean_numeric(assessment.get("tds_base_amount"))
        proposed_amt = _clean_numeric(assessment.get("proposed_tds_amount"))

        if rate is not None and base is not None and proposed_amt is None:
            proposed_amt = round(base * rate / 100.0, 2)

        applicable = assessment.get("tds_applicable")
        if applicable is None:
            applicable = assessment.get("applicable")

        return {
            "tds_applicable": applicable if isinstance(applicable, bool) else (True if rate is not None else False),
            "tds_provision": assessment.get("tds_provision") or assessment.get("tds_section"),
            "tds_rate": rate,
            "tds_base_amount": base,
            "proposed_tds_amount": proposed_amt,
            "tds_needs_review": bool(assessment.get("tds_needs_review", assessment.get("needs_review", False))),
            "tds_reasoning": assessment.get("tds_reasoning") or assessment.get("reason") or "TDS assessment completed.",
        }

    except Exception as exc:
        print(f"[assess_tds_reasoning] Error: {exc}")
        traceback.print_exc()
        return {
            "tds_applicable": None,
            "tds_provision": None,
            "tds_rate": None,
            "tds_base_amount": None,
            "proposed_tds_amount": None,
            "tds_needs_review": True,
            "tds_reasoning": f"TDS assessment fallback due to error: {str(exc)}",
        }

## 7. Unified Pipeline Runner & FastAPI Endpoints

In [ ]:
def analyze_accounting_pipeline(
    invoice_json: dict,
    chart_of_accounts: list = None,
    available_taxes: list = None,
) -> dict:
    """
    Unified entry point executing Task A (COA Classification) and Task B (TDS Assessment)
    with focused inference calls, preserving complete Stage-1 invoice JSON facts.
    """
    if not isinstance(invoice_json, dict):
        raise TypeError("invoice_json must be a dictionary")

    chart_of_accounts = chart_of_accounts or []

    # 1. Task A: COA / Accounting Classification
    accounting_results = classify_coa_accounting(
        complete_invoice_json=invoice_json,
        chart_of_accounts=chart_of_accounts,
    )

    # 2. Task B: TDS Assessment
    tds_result = assess_tds_reasoning(
        complete_invoice_json=invoice_json,
    )

    return {
        "accounting": accounting_results,
        "tds_assessment": tds_result,
    }

In [ ]:
from fastapi import FastAPI, HTTPException
import nest_asyncio
import uvicorn
from pyngrok import ngrok

nest_asyncio.apply()
app = FastAPI(title="Qwen3-4B Accounting & TDS API")

class AccountingRequest(BaseModel):
    invoice_json: Dict[str, Any]
    chart_of_accounts: List[Dict[str, Any]] = Field(default_factory=list)
    available_taxes: List[Dict[str, Any]] = Field(default_factory=list)

@app.get("/health")
async def health():
    return {"status": "ok", "service": "qwen3-4b-accounting"}

@app.post("/api/infer/categorize-accounting")
async def categorize_accounting_endpoint(req: AccountingRequest):
    try:
        return analyze_accounting_pipeline(
            invoice_json=req.invoice_json,
            chart_of_accounts=req.chart_of_accounts,
            available_taxes=req.available_taxes,
        )
    except Exception as exc:
        traceback.print_exc()
        raise HTTPException(status_code=500, detail=str(exc))

## 8. Start ngrok Tunnel & Serve API

In [ ]:
from google.colab import userdata

NGROK_AUTH_TOKEN = userdata.get("NGROK_AUTH_TOKEN")
if NGROK_AUTH_TOKEN:
    ngrok.kill()
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    public_url = ngrok.connect(8000).public_url
    print("Public URL:", public_url)
    print("Health:", public_url + "/health")
    print("Accounting:", public_url + "/api/infer/categorize-accounting")

config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)
await server.serve()